#### **Случайная величина распределена равномерно на отрезке [θ, 2θ]**
**p(x) = 1/θ { [θ, 2θ] }**

In [8]:
import numpy as np


np.random.seed(67)
def get_k_moment_estimation(selection: np.array, k: int):
    return np.sum(selection**k)/(len(selection))

def get_central_k_moment_estimation(selection: np.array, k: int):
    M_s = get_k_moment_estimation(selection, k=1)
    return np.sum((selection-M_s)**k)/(len(selection))

### **Выборка объема n**

In [9]:
n = 100
theta = 10
low = theta
high = 2*theta

x_n = np.random.uniform(low=low, high=high, size=n)

print(*x_n, sep='\n')

15.458514299349787
18.58856610591359
16.858925865665793
13.315918210134342
10.599979295606179
13.86277781042709
12.131498586843556
19.325066500813602
17.228182988558622
10.468953066683795
18.213108493844977
15.918694992908618
12.520715083892394
13.150766903898106
11.878380222591476
13.503199070470592
12.572428118942312
10.11114115323252
14.095885098684473
14.122544515259676
10.43188770896432
19.550087934287223
13.976772847734955
15.296583896437664
12.328129126615163
13.0860495251211
15.519778392267543
17.084756290087437
19.534245531975767
16.019248072519403
14.022515699464202
13.549301995393552
18.141634903058474
12.976418591805718
15.446143249571904
19.630822160313755
11.833954754651208
17.279973382529032
17.332400648741128
10.461399785589261
19.695863937729396
13.705435432006103
17.069442375214336
12.739307015983293
12.49045064855544
13.049723799908493
11.388519886060555
19.751148544073626
10.08411904363797
18.94471800643091
13.5420418023011
19.27035335680679
14.004876677122972
14.68

### **е) Вычислить полученные доверительные интервалы для доверительной вероятности β=0.95**

In [10]:
β = 0.95
x_max = np.max(x_n)

# Точный доверительный интервал
start1 = x_max/(1 + np.pow((1+β)/2, 1/n))
end1 = x_max/(1 + np.pow((1-β)/2, 1/n))

l1 = np.round(end1-start1, 3)
print("Точный доверительный интервал: ")
print(f"{np.round(start1,3)} < θ < {np.round(end1,3)}", end="\n\n")
print("Длина точного доверительного интервала: ")
print(f"l1 = {l1}")

Точный доверительный интервал: 
9.987 < θ < 10.17

Длина точного доверительного интервала: 
l1 = 0.183


In [11]:
a1 = get_k_moment_estimation(x_n, 1)
a2 = get_k_moment_estimation(x_n, 2)

u_0_025 = -1.96
u_0_975 = 1.96

theta_est_m = (2/3)*a1

# Асимптотический доверительный интервал
start2 = theta_est_m - (2*u_0_975*np.sqrt(a2 - a1**2))/(3*np.sqrt(n))
end2 = theta_est_m - (2*u_0_025*np.sqrt(a2 - a1**2))/(3*np.sqrt(n))
l2 = np.round(end2-start2, 3)
print("Асимптотический доверительный интервал: ")
print(f"{np.round(start2,3)} < θ < {np.round(end2,3)}", end="\n\n")
print("Длина асимптотического доверительного интервала: ")
print(f"l2 = {l2}")

Асимптотический доверительный интервал: 
9.402 < θ < 10.169

Длина асимптотического доверительного интервала: 
l2 = 0.767


### **ё) Численно построить бутстраповский доверительный интервал β=0.95**

In [12]:
theta_est_p = ((n+1)/(2*n+1))*x_max
N = 1_000  

def my_function(selection: np.array):
    sel_x_max = np.max(selection)
    theta_est_est_p = ((n+1)/(2*n+1))*sel_x_max
    return theta_est_p - theta_est_est_p


def non_param_bootstrap(selection: np.array, statistic: function, N: int):
    stat_selection = []
    selection_new = np.random.choice(selection, size=len(selection)*N)

    for i in range(0, len(selection)*N, len(selection)):
        stat_selection.append(statistic(selection_new[i:i+len(selection)]))

    return stat_selection

deltas = non_param_bootstrap(x_n, my_function, N)
deltas.sort()
k1 = int((1-β)/2 * N)
k2 = int((1+β)/2 * N)
start3 = theta_est_p - deltas[k2]
end3 = theta_est_p - deltas[k1]
l3 = np.round(end3-start3, 3)
print("Бутстраповский непараметрический доверительный интервал (ОМП): ")
print(f"{np.round(start3,3)} < θ < {np.round(end3,3)}", end="\n\n")
print("Длина бутстраповского непараметрического доверительного интервала (ОМП): ")
print(f"l3 = {l3}")

Бутстраповский непараметрический доверительный интервал (ОМП): 
9.874 < θ < 10.036

Длина бутстраповского непараметрического доверительного интервала (ОМП): 
l3 = 0.161


In [13]:
theta_est_p = ((n+1)/(2*n+1))*x_max
N = 50_000  

def my_model(params: np.array, N:int):
    my_theta = params[0]
    high = my_theta*2
    low = my_theta
    selection = np.random.uniform(low=low, high=high, size=n*N)
    return selection

def my_params(selection: np.array):
    return np.array([((n+1)/(2*n+1))*np.max(selection)])

def param_bootstrap(selection: np.array, statistic: function, model: function, params: function, N: int):
    stat_selection = []
    params_ = params(selection)
    selection_new = my_model(params_, N)

    for i in range(0, len(selection)*N, len(selection)):
        stat_selection.append(statistic(selection_new[i:i+len(selection)]))

    return stat_selection

deltas = param_bootstrap(x_n, my_function, my_model, my_params, N)
deltas.sort()
k1 = int((1-β)/2 * N)
k2 = int((1+β)/2 * N)
start4 = theta_est_p - deltas[k2]
end4 = theta_est_p - deltas[k1]
l4 = np.round(end4-start4, 3)
print("Бутстраповский параметрический доверительный интервал (ОМП): ")
print(f"{np.round(start4,3)} < θ < {np.round(end4,3)}", end="\n\n")
print("Длина бутстраповского доверительного параметрического интервала (ОМП): ")
print(f"l4 = {l4}")

Бутстраповский параметрический доверительный интервал (ОМП): 
9.902 < θ < 10.084

Длина бутстраповского доверительного параметрического интервала (ОМП): 
l4 = 0.182


### **ж) Сравнить все интервалы**

In [14]:

print("Точный доверительный интервал: ")
print(f"{np.round(start1,3)} < θ < {np.round(end1,3)}")
print("Асимптотический доверительный интервал: ")
print(f"{np.round(start2,3)} < θ < {np.round(end2,3)}")
print("Бутстраповский непараметрический доверительный интервал (ОМП): ")
print(f"{np.round(start3,3)} < θ < {np.round(end3,3)}")
print("Бутстраповский параметрический доверительный интервал (ОМП): ")
print(f"{np.round(start4,3)} < θ < {np.round(end4,3)}", end="\n\n")

print("Длина точного доверительного интервала: ")
print(f"l = {l1}")
print("Длина асимптотического доверительного интервала: ")
print(f"l = {l2}")
print("Длина бутстраповского непараметрического доверительного интервала (ОМП): ")
print(f"l3 = {l3}")
print("Длина бутстраповского доверительного параметрического интервала (ОМП): ")
print(f"l4 = {l4}", end="\n\n")

ordered = np.array([
    [l1, "Точный", f"{np.round(start1,3)} < θ < {np.round(end1,3)}"],
    [l2, "Асимптотический", f"{np.round(start2,3)} < θ < {np.round(end2,3)}"],
    [l3, "Бутстраповский непараметрический (ОМП)", f"{np.round(start3,3)} < θ < {np.round(end3,3)}"],
    [l4, "Бутстраповский параметрический (ОМП)", f"{np.round(start4,3)} < θ < {np.round(end4,3)}"],
                ])
winner = ordered[np.argsort(ordered[:,0])[0]]
print("Победитель: ")
print(f"{winner[1]} доверительный интервал:\nl = {winner[0]}")
print(winner[2])

Точный доверительный интервал: 
9.987 < θ < 10.17
Асимптотический доверительный интервал: 
9.402 < θ < 10.169
Бутстраповский непараметрический доверительный интервал (ОМП): 
9.874 < θ < 10.036
Бутстраповский параметрический доверительный интервал (ОМП): 
9.902 < θ < 10.084

Длина точного доверительного интервала: 
l = 0.183
Длина асимптотического доверительного интервала: 
l = 0.767
Длина бутстраповского непараметрического доверительного интервала (ОМП): 
l3 = 0.161
Длина бутстраповского доверительного параметрического интервала (ОМП): 
l4 = 0.182

Победитель: 
Бутстраповский непараметрический (ОМП) доверительный интервал:
l = 0.161
9.874 < θ < 10.036
